# Stage 2: Temporal Modeling & Prediction Head (Transformer)
In this notebook, we implement the Temporal Modeling stage of the Spatio-Temporal SAR Architecture:
1. **Time-Series Sequences**: We adapt our dataset to return sequences of length $T$ (simulating temporal data).
2. **Temporal Modeling (Transformer)**: A Transformer Encoder architecture that takes latent sequences from the pre-trained Autoencoder.
3. **Prediction Head**: A multi-output head that translates the temporal features into Water Body Segmentation Masks.
4. **Training**: End-to-end training of the temporal model.

In [1]:
import os
import torch
from torch.utils.data import DataLoader
from dataset import TrueTimeSeriesSARDataset, get_train_val_test_splits
from models import TemporalTransformerPredictor
from train import train_temporal_model
from utils import set_seed, load_config

# 1. Setup & Configuration
config = load_config('config.json')
set_seed(config['training']['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


In [2]:
# 2. Data Loading (Temporal Sequences)
s1_dir = config['data']['s1_dir']
label_dir = config['data']['label_dir']

# In a true temporal dataset, we expect sequences organized by location.
# As a fallback for Sen1Floods11 single snapshots, we will simulate the grouping 
# just to test the pipeline, but the TrueTimeSeriesSARDataset class is built for real sequences.

s1_files = sorted([f for f in os.listdir(s1_dir) if f.endswith('.tif')]) if os.path.exists(s1_dir) else []
label_files = sorted([f for f in os.listdir(label_dir) if f.endswith('.tif')]) if os.path.exists(label_dir) else []

# Creating dummy sequences of length N for the single files available to test the pipeline
seq_length = config['data']['seq_length']
sequences = []
if len(s1_files) >= seq_length:
    for i in range(len(s1_files) - seq_length + 1):
        s1_seq = [os.path.join(s1_dir, s1_files[i+j]) for j in range(seq_length)]
        label_path = os.path.join(label_dir, label_files[i+seq_length-1])
        sequences.append((s1_seq, label_path))

if len(sequences) > 0:
    train_idx, val_idx, test_idx = get_train_val_test_splits(len(sequences), config['data']['split_ratio'])
    
    train_seqs = [sequences[i] for i in train_idx]
    val_seqs = [sequences[i] for i in val_idx]
    
    train_dataset = TrueTimeSeriesSARDataset(train_seqs, seq_length=seq_length, target_size=tuple(config['data']['target_size']))
    val_dataset = TrueTimeSeriesSARDataset(val_seqs, seq_length=seq_length, target_size=tuple(config['data']['target_size']))
    
    train_loader = DataLoader(train_dataset, batch_size=config['training']['batch_size'], shuffle=True, num_workers=config['data']['num_workers'])
    val_loader = DataLoader(val_dataset, batch_size=config['training']['batch_size'], shuffle=False, num_workers=config['data']['num_workers'])
    
    print(f"Loaded {len(train_dataset)} training and {len(val_dataset)} validation sequences.")
else:
    print("Not enough data to create temporal sequences.")


Loaded 26 training and 5 validation sequences.


In [3]:
# 3. Model Definition & Training
if len(sequences) > 0:
    model = TemporalTransformerPredictor(
        seq_length=config['data']['seq_length'],
        embed_dim=config['model']['transformer_embed_dim'],
        num_heads=config['model']['transformer_heads'],
        hidden_dim=config['model']['transformer_hidden_dim'],
        num_layers=config['model']['transformer_layers'],
        dropout=config['model']['dropout_rate']
    ).to(device)
    
    # Load pretrained spatial encoder weights if available
    encoder_path = 'models/encoder.pth'
    if os.path.exists(encoder_path):
        print("Loading pretrained spatial encoder weights...")
        try:
            model.spatial_encoder.encoder.load_state_dict(torch.load(encoder_path, weights_only=True))
        except Exception as e:
            print(f"Could not load encoder weights: {e}")
    else:
        print("Pretrained encoder not found. Training from scratch.")
        
    train_temporal_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=config['training']['temporal_epochs'],
        lr=config['training']['temporal_lr'],
        device=device
    )


Loading pretrained spatial encoder weights...


Starting Temporal Transformer Training...


Epoch [1/30], Train Loss: 0.7909, Val Loss: 0.7290
Val Metrics - IoU: 0.0192, Acc: 0.0192, F1: 0.0376, P: 0.0192, R: 1.0000


Epoch [2/30], Train Loss: 0.7761, Val Loss: 0.7230
Val Metrics - IoU: 0.0192, Acc: 0.0209, F1: 0.0376, P: 0.0192, R: 0.9982


Epoch [3/30], Train Loss: 0.7612, Val Loss: 0.7172
Val Metrics - IoU: 0.0193, Acc: 0.1513, F1: 0.0378, P: 0.0193, R: 0.8705


Epoch [4/30], Train Loss: 0.7477, Val Loss: 0.7115
Val Metrics - IoU: 0.0190, Acc: 0.3718, F1: 0.0372, P: 0.0192, R: 0.6338


Epoch [5/30], Train Loss: 0.7323, Val Loss: 0.7058
Val Metrics - IoU: 0.0187, Acc: 0.4689, F1: 0.0367, P: 0.0190, R: 0.5288


Epoch [6/30], Train Loss: 0.7130, Val Loss: 0.7002
Val Metrics - IoU: 0.0186, Acc: 0.5169, F1: 0.0365, P: 0.0190, R: 0.4784


Epoch [7/30], Train Loss: 0.6988, Val Loss: 0.6944
Val Metrics - IoU: 0.0187, Acc: 0.5409, F1: 0.0368, P: 0.0191, R: 0.4574


Epoch [8/30], Train Loss: 0.6783, Val Loss: 0.6889
Val Metrics - IoU: 0.0186, Acc: 0.5534, F1: 0.0365, P: 0.0190, R: 0.4417
Early stopping triggered based on IoU.
Temporal Training Complete. Best model saved to models/best_temporal_model.pth
